# ONNX Runtime CPU EP Performance Analysis

**Objective:** Compare ORT CPU EP latency across quantization formats (MNB native, QDQ fused, QDQ unfused), weight layouts (original vs transposed), bit widths (4, 8), symmetry/signedness/granularity variants across devices (AMD, Intel, ARM).

**Devices:**
| Label | CPU | Notes |
|-------|-----|-------|
| `amd` | AMD64 Family 26 Model 36 (HP AMD laptop) | ORT 1.24.2, 10 iterations |
| `intel` | Intel64 Family 6 Model 186 (Surface Laptop) | ORT 1.24.1, 10 iterations |
| `arm` | ARM64 (Surface ARM laptop) | ORT 1.24.1, 50 iterations |

**Notes:**
- ORT only fuses DQ+MatMul → MatMulNBits for **4-bit block** quantization 
- Transpose models add a Transpose between DQ and MatMul to test column-major weight layout
- 8 models failed to load for `channel-sym-signed-transpose` variants (possible ORT bug)
- ARM data seem to have higher variance and maybe some memory pressure issues for 7b-8 bit models. Might need to examine/rerun ARM test

In [165]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

# --- Load all three devices ---
DATA_DIR = Path(r"C:\dev\cpu_perf")
DEVICES = {
    "amd": DATA_DIR / "hp_amd_laptop_perf",
    "intel": DATA_DIR / "surface_laptop_perf",
    "arm": DATA_DIR / "surface_arm_laptop_perf",
}

frames = []
for device, path in DEVICES.items():
    csv_path = path / "summary.csv"
    if not csv_path.exists():
        print(f"WARNING: {csv_path} not found, skipping")
        continue
    tmp = pd.read_csv(csv_path)
    tmp["device"] = device
    frames.append(tmp)
    print(f"  {device}: {len(tmp)} rows from {csv_path.name}")

df = pd.concat(frames, ignore_index=True)

# Type coercion
df["bits"] = df["bits"].astype(int)
df["seq_length"] = df["seq_length"].astype(int)
df["mean_ms"] = df["mean_ms"].astype(float)
df["model_load_time_s"] = pd.to_numeric(df["model_load_time_s"], errors="coerce")

print(f"\nTotal rows: {len(df)}")
print(f"Devices: {list(df['device'].unique())}")
print(f"Weight layouts: {list(df['weight_layout'].unique())}")
print(f"Scenarios: {list(df['scenario'].unique())}")
print(f"Seq lengths: {sorted(int(x) for x in df['seq_length'].unique())}")
print(f"Bits: {sorted(int(x) for x in df['bits'].unique())}")
print(f"Model sizes: {list(df['model_size'].unique())}")

  amd: 640 rows from summary.csv
  intel: 640 rows from summary.csv
  arm: 560 rows from summary.csv

Total rows: 1840
Devices: ['amd', 'intel', 'arm']
Weight layouts: ['original', 'transpose']
Scenarios: ['native', 'qdq_fused', 'qdq_unfused']
Seq lengths: [1, 128, 256, 512, 1024]
Bits: [4, 8]
Model sizes: ['0.5b', '1.5b', '3b', '7b']


In [166]:
# Save CSV
out_path = Path("all_devices_summary.csv")
mnb_mask = df["format_type"] == "mnb"
df.loc[mnb_mask, "granularity"] = "n/a"
df.loc[mnb_mask, "signedness"] = "n/a"
print(f"Set {mnb_mask.sum()} MNB rows: granularity='n/a', signedness='n/a'")

# Re-save CSV
df.to_csv(out_path, index=False)
print(f"Updated {out_path}")

Set 240 MNB rows: granularity='n/a', signedness='n/a'
Updated all_devices_summary.csv


In [143]:
# --- Helper functions ---
MODEL_SIZE_ORDER = ["0.5b", "1.5b", "3b", "7b"]
SEQ_ORDER = [1, 128, 256, 512, 1024]
DEVICE_ORDER = ["amd", "intel", "arm"]

def pivot_latency(data, index_cols, columns_col, values_col="mean_ms"):
    """Pivot data into a wide table with mean_ms values."""
    return data.pivot_table(
        index=index_cols, columns=columns_col,
        values=values_col, aggfunc="first"
    )


def ratio_table(data, col_a, col_b, label_a="A", label_b="B"):
    """Compute ratio B/A for two subsets identified by a column value."""
    a = data[data[col_a] == label_a].copy()
    b = data[data[col_a] == label_b].copy()
    merge_cols = [c for c in a.columns if c not in ["mean_ms", col_a, "model_load_time_s"]]
    merged = a.merge(b, on=merge_cols, suffixes=(f"_{label_a}", f"_{label_b}"))
    merged["ratio"] = merged[f"mean_ms_{label_b}"] / merged[f"mean_ms_{label_a}"]
    return merged


def comparison_table(data, group_cols, compare_col, compare_values, metric="mean_ms"):
    """Build a side-by-side comparison table.

    Returns a DataFrame with one row per (group_cols) combination and
    one column per compare_value, plus a ratio column if exactly 2 compare_values.
    """
    tables = {}
    for val in compare_values:
        subset = data[data[compare_col] == val].set_index(group_cols)[metric]
        tables[val] = subset

    result = pd.DataFrame(tables)
    if len(compare_values) == 2:
        a, b = compare_values
        result[f"{b}/{a}"] = result[b] / result[a]
    return result


def display_comparison(title, data, group_cols, compare_col, compare_values, metric="mean_ms"):
    """Print a comparison table with a title."""
    print(f"\n{'='*80}")
    print(f"  {title}")
    print(f"{'='*80}")
    tbl = comparison_table(data, group_cols, compare_col, compare_values, metric)
    print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))
    return tbl



## A. MNB Native vs QDQ Fused vs QDQ Unfused — 4-bit block quantization



**Expected**: MNB ≈ QDQ fused << QDQ unfused


In [161]:

# Filter to 4-bit, block-quantized, original layout, asym signed (most common LLM config)
mnb_4 = df[(df["format_type"] == "mnb") & (df["bits"] == 4) & (df["weight_layout"] == "original")].copy()
qdq_fused_4 = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["weight_layout"] == "original") & (df["scenario"] == "qdq_fused") & (df["signedness"] == "signed")
].copy()
qdq_unfused_4 = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["weight_layout"] == "original") & (df["scenario"] == "qdq_unfused") & (df["signedness"] == "signed")
].copy()

# Build merged comparison table (asym only — sym behaves identically)
mnb_sub = mnb_4[mnb_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "mnb_ms"})
fused_sub = qdq_fused_4[qdq_fused_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "fused_ms"})
unfused_sub = qdq_unfused_4[qdq_unfused_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "unfused_ms"})

merged = mnb_sub.merge(fused_sub, on=["device", "model_size", "seq_length"], how="outer")
merged = merged.merge(unfused_sub, on=["device", "model_size", "seq_length"], how="outer")
merged["fused/mnb"] = merged["fused_ms"] / merged["mnb_ms"]
merged["unfused/mnb"] = merged["unfused_ms"] / merged["mnb_ms"]
merged = merged.set_index(["device", "model_size", "seq_length"]).sort_index()

# --- Table 1: Latency (ms) — seq=1 (decode) and seq=1024 (prefill) ---
print("  Table 1: Latency comparison (ms) — 4-bit block asym signed, original layout")
print("  " + "-"*85)
for seq in [1, 1024]:
    label = "Decode (seq=1)" if seq == 1 else "Prefill (seq=1024)"
    sub = merged.xs(seq, level="seq_length")[["mnb_ms", "fused_ms", "unfused_ms", "fused/mnb", "unfused/mnb"]]
    print(f"\n  {label}:")
    print(sub.to_string(float_format=lambda x: f"{x:.1f}"))

# --- Table 2: Model load times ---
mnb_load = mnb_4[mnb_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "mnb_s"})
fused_load = qdq_fused_4[qdq_fused_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "fused_s"})
unfused_load = qdq_unfused_4[qdq_unfused_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "unfused_s"})
load_times = mnb_load.merge(fused_load, on=["device", "model_size"]).merge(unfused_load, on=["device", "model_size"])
load_times["fused/mnb"] = load_times["fused_s"] / load_times["mnb_s"]

print(f"\n\n  Table 2: Model load times (s) — includes PrePack for MNB & fused; unfused skips PrePack")
print("  " + "-"*85)
print(load_times.set_index(["device", "model_size"]).sort_index().to_string(float_format=lambda x: f"{x:.2f}"))


  Table 1: Latency comparison (ms) — 4-bit block asym signed, original layout
  -------------------------------------------------------------------------------------

  Decode (seq=1):
                   mnb_ms  fused_ms  unfused_ms  fused/mnb  unfused/mnb
device model_size                                                      
amd    0.5b           7.1       8.3       691.9        1.2         97.2
       1.5b          19.2      20.6      2267.2        1.1        118.3
       3b            46.6      41.0      4388.0        0.9         94.2
       7b            98.1      78.8      8121.9        0.8         82.8
arm    0.5b           9.7      11.3       624.2        1.2         64.4
       1.5b          87.9      33.6      1792.3        0.4         20.4
       3b           116.5      56.4      3461.7        0.5         29.7
       7b           112.0      59.1     10612.2        0.5         94.8
intel  0.5b           8.9       8.4       984.2        0.9        110.7
       1.5b          22

In [69]:

# --- Table 3: Estimated DequantizeLinear overhead (unfused - fused) at seq=1 ---
# At seq=1 the MatMul compute is trivial, so the delta ≈ pure DQ cost
m = merged.copy()
m["dq_overhead_ms"] = m["unfused_ms"] - m["fused_ms"]

seq1 = m.xs(1, level="seq_length")[["fused_ms", "unfused_ms", "dq_overhead_ms"]]

print("  Table 3: Estimated DequantizeLinear cost at seq=1 (ms)")
print("  At decode (seq=1), MatMul compute is trivial → delta ≈ pure DQ overhead")
print("  " + "-"*70)
print(seq1.to_string(float_format=lambda x: f"{x:.0f}"))

print("\n\n  DQ cost scaling relative to 0.5b (should be ~linear with param count):")
for dev in DEVICE_ORDER:
    dd = seq1.xs(dev, level="device")["dq_overhead_ms"]
    base = dd.iloc[0]
    print(f"    {dev}: " + "  ".join(f"{sz}={v/base:.1f}x" for sz, v in dd.items()))


  Table 3: Estimated DequantizeLinear cost at seq=1 (ms)
  At decode (seq=1), MatMul compute is trivial → delta ≈ pure DQ overhead
  ----------------------------------------------------------------------
                   fused_ms  unfused_ms  dq_overhead_ms
device model_size                                      
amd    0.5b               8         692             684
       1.5b              21        2267            2247
       3b                41        4388            4347
       7b                79        8122            8043
arm    0.5b              11         624             613
       1.5b              34        1792            1759
       3b                56        3462            3405
       7b                59       10612           10553
intel  0.5b               8         984             976
       1.5b              23        3190            3168
       3b                44        6257            6212
       7b               101       14140           14039


  DQ cost 

---

#### I asked copilot to look deeper into itidentified following Optimization Opportunities for the Unfused DequantizeLinear Path

Optimization 1 (easy): Add threading to the blocked sub-byte path

The non-sub-byte DQ already has threading (gated behind `ORT_CLIENT_PACKAGE_BUILD` at line 340):
```cpp
ParDequantizeLinearStd<T>(input, output, N, scale[k], zero_point ? zero_point[k] : 0, thread_pool);
```
This calls `MlasTryBatchParallel` internally to split N elements across threads. But the blocked sub-byte path at line 451 explicitly ignores the thread pool.

**What to do**: Replace `ORT_UNUSED_PARAMETER(thread_pool)` at line 452 with `concurrency::ThreadPool::TryParallelFor` to partition the outer `M × num_blocks` loop across threads. The inner loop body has no data dependencies between blocks, so this is safe. This is the single most impactful change — it immediately scales with available cores.

Expected impact: **~4–8× speedup on multi-core systems** (proportional to core count).

Optimization 2 (medium): Call `MlasDequantizeBlockwise` (it already exists and is threaded)

MLAS already has a threaded dequantization function at `core/mlas/lib/q4_dq.cpp:1847`:
```cpp
template <typename ElementT, int qbits>
void MlasDequantizeBlockwise(ElementT* dst, const uint8_t* src, const ElementT* scales,
    const uint8_t* zero_points, int block_size, bool columnwise,
    int rows, int columns, MLAS_THREADPOOL* thread_pool);
```

`MatMulNBits::ComputeBUnpacked()` already calls it at `matmul_nbits.cc:599`:
```cpp
MlasDequantizeBlockwise<float, 4>(tmp_b_data_ptr.get(), b_data, scales_data,
    static_cast<const uint8_t*>(zero_points_data),
    static_cast<int32_t>(block_size_), column_wise_quant_,
    static_cast<int32_t>(K_), static_cast<int32_t>(N_), thread_pool);
```

**Blocker**: Layout mismatch. `MlasDequantizeBlockwise` expects **column-major** packed weights (for the MatMulNBits weight format). DequantizeLinear uses **row-major** packed weights (ONNX QDQ spec). From `q4_dq.cpp:391`: `"Source is row major. Dest, scale and zp are column major."`

**What to do**: Write a new MLAS function `MlasDequantizeBlockwiseRowMajor()` that keeps the threading (`MlasTryBatchParallel`) but adds SIMD nibble extraction for row-major layout. The SIMD patterns already exist in ORT:
- **AVX2** (`sqnbitgemm_kernel_avx2.cpp:265`): `_mm256_and_si256(data, _mm256_set1_epi8(0xF))` — extracts 32 nibbles per instruction
- **NEON** (`sqnbitgemm_kernel_neon_fp32.cpp:156`): `vand_u8(data, vdup_n_u8(0x0F))` + `vshr_n_u8(data, 4)` — 8+8 nibbles per instruction

Expected impact: **Another 4–8× on top of threading** from SIMD (32 elements/cycle on AVX2 vs 1 scalar).

Optimization 3 (medium): Add `PrePack()` for constant weight caching

`DequantizeLinear` has **no `PrePack()`** (it's a plain `OpKernel`, line 18). Every inference call re-dequantizes constant weights from scratch. Meanwhile, `MatMulNBits` has a full `PrePack()` at `matmul_nbits.cc:188` that pre-packs weights at load time.

**What to do**: Add a `PrePack()` override to `DequantizeLinear` that:
1. Checks if inputs 0 (data), 1 (scale), 2 (zero_point) are constant initializers
2. If yes, dequantizes once at load time and caches the float32 result
3. In `Compute()`, just `memcpy` the cached buffer to the output

For LLM weight dequantization, all inputs are always constant. This eliminates **100% of the per-inference DQ cost** — the only remaining cost is the memcpy.

Expected impact: **Eliminates repeat dequant entirely** (~10ms → ~1ms per DQ node from pure memcpy). Trade-off: higher memory usage (stores the full fp32 weights alongside int4).

Optimization 4 (easy): Remove the `ORT_CLIENT_PACKAGE_BUILD` gate

The non-sub-byte per-axis DQ threading/SIMD at line 340 is gated behind `#if defined(ORT_CLIENT_PACKAGE_BUILD)` with a TODO: *"Make this the default behavior after more testing."* 

This means that standard ORT builds don't get threading for int8/uint8 DequantizeLinear either. Removing this gate (after validation) would benefit all DQ users.

#### Summary: Effort vs Impact

| Optimization | Effort | Expected Speedup | File(s) |
|---|---|---|---|
| 1. Add threading | Low | ~4–8× | `quantize_linear.cc:451` |
| 2. SIMD nibble extraction | Medium | ~4–8× (on top of #1) | New MLAS fn + `quantize_linear.cc` |
| 3. PrePack for const weights | Medium | Eliminates repeat DQ | `quantize_linear.cc:18` |
| 4. Remove client build gate | Low | Enables int8 DQ threading | `quantize_linear.cc:340` |
| **1+2 combined** | **Medium** | **~16–60×** | — |

Optimizations 1+2 together could close most of the 80–150× gap vs fused. Optimization 3 would eliminate the gap entirely for constant weights (the common LLM case).


### A.2 MNB 8-bit vs QDQ 8-bit Unfused

Section A quantifies MNB vs fused vs unfused for 4-bit only. Since 8-bit QDQ never fuses,
the gap between MNB 8-bit (native with PrePack + MLAS SGEMM) and QDQ 8-bit block unfused
(scalar DQ + MatMul) is the optimization opportunity for 8-bit.

In [ ]:
# A.2: MNB 8-bit vs QDQ 8-bit unfused
mnb_8 = df[
    (df["format_type"] == "mnb") & (df["bits"] == 8) &
    (df["weight_layout"] == "original") & (df["symmetry"] == "sym")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "mnb_8b_ms"})

qdq_8_uf = df[
    (df["format_type"] == "qdq") & (df["bits"] == 8) & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_fused") & (df["weight_layout"] == "original") &
    (df["symmetry"] == "sym") & (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "qdq_8b_ms"})

gap_8 = mnb_8.merge(qdq_8_uf, on=["device", "model_size", "seq_length"])
gap_8["qdq/mnb"] = gap_8["qdq_8b_ms"] / gap_8["mnb_8b_ms"]

print("  A.2: MNB 8-bit native vs QDQ 8-bit block unfused (sym, signed, original)")
print("  " + "-"*85)
for seq in [1, 1024]:
    label = "Decode (seq=1)" if seq == 1 else "Prefill (seq=1024)"
    sub = gap_8[gap_8["seq_length"] == seq].set_index(["device", "model_size"]).sort_index()
    print(f"\n  {label}:")
    print(sub[["mnb_8b_ms", "qdq_8b_ms", "qdq/mnb"]].to_string(float_format=lambda x: f"{x:.1f}"))

print(f"\n  Per-device qdq/mnb ratio summary (8-bit):")
for dev in DEVICE_ORDER:
    ratios = gap_8[gap_8["device"] == dev]["qdq/mnb"]
    if not ratios.empty:
        print(f"    {dev}: min={ratios.min():.1f}x  max={ratios.max():.1f}x  median={ratios.median():.1f}x")
seq1_8 = gap_8[gap_8["seq_length"] == 1]
print(f"\n  Compare: 4-bit unfused/mnb at seq=1: 80-154x")
print(f"  Compare: 8-bit unfused/mnb at seq=1: {seq1_8['qdq/mnb'].min():.0f}-{seq1_8['qdq/mnb'].max():.0f}x")
print(f"  8-bit gap is smaller because 8-bit DQ avoids nibble extraction.")
print(f"  But still substantial: extending fusion to 8-bit would close it.")

## B. MNB Symmetric vs Asymmetric

Check if asymmetric (with zero-point) adds overhead vs symmetric.

In [77]:
mnb = df[(df["format_type"] == "mnb") & (df["weight_layout"] == "original")].copy()

for bits in [4, 8]:
    sub = mnb[mnb["bits"] == bits]
    tbl = comparison_table(
        sub,
        group_cols=["device", "model_size", "seq_length"],
        compare_col="symmetry",
        compare_values=["sym", "asym"],
    )
    tbl.columns = ["sym", "asym", "asym/sym"]

    print(f"\n{'='*90}")
    print(f"  B. MNB sym vs asym — {bits}-bit (ratio = asym/sym)")
    print(f"{'='*90}")
    print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

    # Summary stats per device
    print(f"\n  Summary:")
    for dev in DEVICE_ORDER:
        ratios = tbl.loc[dev]["asym/sym"]
        print(f"    {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  "
              f"median={ratios.median():.2f}  mean={ratios.mean():.2f}  std={ratios.std():.2f}")

# ARM 4-bit: check if the difference is systematic per model size
print(f"\n{'='*90}")
print("  ARM 4-bit asym/sym by model_size")
print(f"{'='*90}")
arm_4 = mnb[(mnb["device"] == "arm") & (mnb["bits"] == 4)]
tbl_arm = comparison_table(arm_4, ["model_size", "seq_length"], "symmetry", ["sym", "asym"])
tbl_arm.columns = ["sym", "asym", "asym/sym"]
for ms in MODEL_SIZE_ORDER:
    if ms not in tbl_arm.index.get_level_values(0):
        continue
    ratios = tbl_arm.loc[ms]["asym/sym"]
    mean_r = ratios.mean()
    direction = "asym SLOWER" if mean_r > 1.05 else ("asym FASTER" if mean_r < 0.95 else "~EQUAL")
    print(f"  {ms}: mean={mean_r:.2f} ({direction})  per-seq={[f'{v:.2f}' for v in ratios.values]}")


  B. MNB sym vs asym — 4-bit (ratio = asym/sym)
                                  sym     asym  asym/sym
device model_size seq_length                            
amd    0.5b       1              7.05     7.12      1.01
                  128          173.69   131.98      0.76
                  256          304.46   269.79      0.89
                  512          690.23   582.99      0.84
                  1024        1459.12  1386.95      0.95
       1.5b       1             20.75    19.17      0.92
                  128          467.04   411.74      0.88
                  256          965.10   837.89      0.87
                  512         2010.46  1775.49      0.88
                  1024        6192.57  4053.75      0.65
       3b         1             46.51    46.57      1.00
                  128         1405.70  1447.67      1.03
                  256         2793.26  2873.39      1.03
                  512         5672.13  5771.73      1.02
                  1024       12135.81 1

### Observations

1. Intel: sym ≈ asym across both bit widths. Median ratio 1.00 (4-bit) and 0.99 (8-bit) as expected
2. AMD 8-bit: median 1.02, range 0.94–1.16 as expected

3. AMD 4-bit: median 0.98, but a few outliers where asym is faster (1.5b seq=1024: 0.65, 0.5b seq=128: 0.76). Likely measurement noise worth re-running with more iterations to confirm.

4. ARM 4-bit ≥1.5b: asym is systematically ~50% slower. This is consistent across all seq lengths for each model size. Too uniform to be scheduling noise 

5. ARM 0.5b 4-bit: opposite direction (mean 0.73, asym faster). The sym seq=1 value of 29ms vs asym 10ms looks suspicious and may need a rerun.

6. ARM 8-bit: erratic ratios (0.50–2.83), no clear pattern 

Conclusion: sym vs asym is a non-factor on x86. ARM shows a real asym penalty at 4-bit for models ≥1.5b needs investigation into the ARM kernel's zero-point handling.

### B.2 QDQ Sym vs Asym within the Unfused Path

Section B covers MNB sym vs asym. Here we compare within QDQ unfused only,
where the scalar DQ kernel must subtract zero-point for asymmetric.
The cost should be measurable in the unoptimized code path.

In [ ]:
# B.2: QDQ unfused sym vs asym
# For 4-bit: use qdq_unfused (qdq_fused actually fuses to MatMulNBits)
# For 8-bit: use qdq_fused (8-bit never fuses, so it's effectively unfused)
qdq_uf_sym = pd.concat([
    df[(df["format_type"] == "qdq") & (df["bits"] == 4) &
       (df["scenario"] == "qdq_unfused") & (df["granularity"] == "block") &
       (df["weight_layout"] == "original") & (df["symmetry"] == "sym") &
       (df["signedness"] == "signed")],
    df[(df["format_type"] == "qdq") & (df["bits"] == 8) &
       (df["scenario"] == "qdq_fused") & (df["granularity"] == "block") &
       (df["weight_layout"] == "original") & (df["symmetry"] == "sym") &
       (df["signedness"] == "signed")]
])[["device", "model_size", "bits", "seq_length", "mean_ms"]].rename(
    columns={"mean_ms": "sym_ms"})

qdq_uf_asym = pd.concat([
    df[(df["format_type"] == "qdq") & (df["bits"] == 4) &
       (df["scenario"] == "qdq_unfused") & (df["granularity"] == "block") &
       (df["weight_layout"] == "original") & (df["symmetry"] == "asym") &
       (df["signedness"] == "signed")],
    df[(df["format_type"] == "qdq") & (df["bits"] == 8) &
       (df["scenario"] == "qdq_fused") & (df["granularity"] == "block") &
       (df["weight_layout"] == "original") & (df["symmetry"] == "asym") &
       (df["signedness"] == "signed")]
])[["device", "model_size", "bits", "seq_length", "mean_ms"]].rename(
    columns={"mean_ms": "asym_ms"})

b2 = qdq_uf_sym.merge(qdq_uf_asym, on=["device", "model_size", "bits", "seq_length"])
b2["asym/sym"] = b2["asym_ms"] / b2["sym_ms"]

print("  B.2: QDQ unfused sym vs asym (block, original, signed)")
print("  " + "-"*70)
for bits_val in [4, 8]:
    sub = b2[(b2["bits"] == bits_val) & (b2["seq_length"] == 1)]
    if sub.empty:
        continue
    print(f"\n  {bits_val}-bit, Decode (seq=1):")
    pivot = sub.pivot_table(index="model_size", columns="device",
                            values="asym/sym", aggfunc="first")
    pivot = pivot.reindex(columns=DEVICE_ORDER)
    print(pivot.to_string(float_format=lambda x: f"{x:.2f}"))

print("\n  Key: asym/sym > 1.0 means asym is slower (zero-point subtraction overhead)")
print("  In unfused path, the zero-point cost is exposed because it's scalar code.")
print("  In fused path (MNB), the MLAS kernel absorbs zero-point into packed weights.")

## C. MNB 4-bit vs 8-bit

Quantify latency difference between 4-bit and 8-bit. 4-bit should be faster at decode (less bandwidth). At prefill, compute dominates so difference should be small.

In [88]:
# Section C: MNB 4-bit vs 8-bit
# Section B showed sym ≈ asym on x86. Use sym here (cleaner ARM data).
mnb = df[(df["format_type"] == "mnb") & (df["weight_layout"] == "original")].copy()
mnb_sym = mnb[mnb["symmetry"] == "sym"]

tbl = comparison_table(
    mnb_sym,
    group_cols=["device", "model_size", "seq_length"],
    compare_col="bits",
    compare_values=[4, 8],
)
tbl.columns = ["4-bit", "8-bit", "8b/4b"]

print(f"{'='*80}")
print(f"  C. MNB 4-bit vs 8-bit — sym, original layout (ratio = 8b/4b)")
print(f"{'='*80}")
print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

# Verify asym ratios match on x86 (ARM asym is noisy per Section B)
tbl_asym = comparison_table(
    mnb[(mnb["symmetry"] == "asym") & (mnb["device"].isin(["amd", "intel"]))],
    group_cols=["device", "model_size", "seq_length"],
    compare_col="bits",
    compare_values=[4, 8],
)
tbl_asym.columns = ["4-bit", "8-bit", "8b/4b"]
ratio_diff = (tbl_asym["8b/4b"] - tbl.loc[["amd", "intel"]]["8b/4b"]).abs()
print(f"\nAsym vs sym ratio check (x86): max diff = {ratio_diff.max():.2f}, mean = {ratio_diff.mean():.2f}")
print(f"  (AMD 1.5b outlier from Section B accounts for max diff)")

  C. MNB 4-bit vs 8-bit — sym, original layout (ratio = 8b/4b)
                                4-bit    8-bit  8b/4b
device model_size seq_length                         
amd    0.5b       1              7.05    10.11   1.43
                  128          173.69   146.08   0.84
                  256          304.46   273.11   0.90
                  512          690.23   567.94   0.82
                  1024        1459.12  1305.17   0.89
       1.5b       1             20.75    39.64   1.91
                  128          467.04   642.76   1.38
                  256          965.10  1307.94   1.36
                  512         2010.46  2883.31   1.43
                  1024        6192.57  6459.83   1.04
       3b         1             46.51    73.19   1.57
                  128         1405.70  1292.49   0.92
                  256         2793.26  2582.51   0.92
                  512         5672.13  5410.64   0.95
                  1024       12135.81 10884.73   0.90
       7b         1

### Observations

1. x86 seq=1: 8b/4b = 1.4-2.0x. Expected. Bandwidth-bound, 4-bit weights are half the size.

2. Intel seq>=128: 8b/4b = 0.88-1.08x. Bit width barely matters at prefill (compute-bound). 7b shows slight 8-bit advantage (mean 0.93x), rest are ~1.0x.

3. AMD seq>=128: 0.5b (mean 0.86x), 3b (0.92x), 7b (0.91x) all show 8-bit slightly faster. But AMD 1.5b is an outlier: 8-bit is 1.04-1.43x slower (mean 1.30x). Same anomalous 1.5b data point from Section B. Not seen on Intel (1.5b mean 1.03x). Likely a measurement issue, worth re-running.

4. ARM 0.5b-3b: 8-bit is faster than 4-bit at seq>=128 (0.48-0.92x). Stronger effect than x86. Possibly related to 4-bit nibble unpacking overhead or a less optimized ARM 4-bit kernel. Needs profiling to confirm root cause.

5. ARM 7b-8bit: 1.6-4.3x across all seq lengths. Model exceeds 16GB device RAM, causes swapping. Not usable.

6. Sym vs asym: 8b/4b ratios consistent on x86 (AMD 1.5b outlier from Section B aside). ARM asym data noisy as before.

## D. QDQ Signed vs Unsigned Asymmetric

Check if signed vs unsigned affects QDQ performance. 4-bit fuses to MatMulNBits, 8-bit does not. Expected: ~1.0x.

In [103]:
# Section D: QDQ signed vs unsigned — asymmetric, block, original layout
# 4-bit: check both fused (MatMulNBits) and unfused (DequantizeLinear) paths
# 8-bit: only fused scenario (which is actually unfused — 8-bit doesn't fuse)

qdq_asym_block_orig = df[
    (df["format_type"] == "qdq") & (df["symmetry"] == "asym") & (df["granularity"] == "block") &
    (df["weight_layout"] == "original")
].copy()

configs = [
    (4, "qdq_fused",   "4-bit fused (MatMulNBits kernel)"),
    (4, "qdq_unfused", "4-bit unfused (DequantizeLinear + MatMul)"),
    (8, "qdq_fused",   "8-bit (no fusion — runs DequantizeLinear + MatMul)"),
]

for bits, scenario, label in configs:
    sub = qdq_asym_block_orig[(qdq_asym_block_orig["bits"] == bits) & (qdq_asym_block_orig["scenario"] == scenario)]
    tbl = comparison_table(
        sub,
        group_cols=["device", "model_size", "seq_length"],
        compare_col="signedness",
        compare_values=["signed", "unsigned"],
    )
    tbl.columns = ["signed", "unsigned", "u/s"]

    print(f"\n{'='*90}")
    print(f"  D. {label} — unsigned/signed ratio")
    print(f"{'='*90}")
    for dev in DEVICE_ORDER:
        try:
            ratios = tbl.loc[dev]["u/s"]
            print(f"  {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  median={ratios.median():.2f}")
        except KeyError:
            pass
    # Show outliers
    outliers = tbl[(tbl["u/s"] < 0.85) | (tbl["u/s"] > 1.15)]
    if not outliers.empty:
        print(f"  Outliers (>15% diff):")
        print(outliers.to_string(float_format=lambda x: f"{x:.2f}"))



  D. 4-bit fused (MatMulNBits kernel) — unsigned/signed ratio
  amd: min=0.91  max=1.05  median=1.00
  intel: min=0.92  max=1.13  median=1.01
  arm: min=0.88  max=1.75  median=1.00
  Outliers (>15% diff):
                              signed  unsigned  u/s
device model_size seq_length                       
arm    0.5b       1            11.32     19.76 1.75
                  128         268.96    381.43 1.42

  D. 4-bit unfused (DequantizeLinear + MatMul) — unsigned/signed ratio
  amd: min=0.85  max=0.98  median=0.92
  intel: min=0.79  max=0.98  median=0.90
  arm: min=0.83  max=1.13  median=1.00
  Outliers (>15% diff):
                               signed  unsigned  u/s
device model_size seq_length                        
intel  0.5b       1            984.20    773.62 0.79
                  128         1275.98   1051.81 0.82
       1.5b       1           3190.45   2698.07 0.85
       7b         1          14139.84  11965.76 0.85
                  128        18216.17  15296.36 0.84


### Observations

1. 4-bit fused: signed = unsigned on x86. AMD median 1.00 (0.91-1.05). Intel median 1.01 (0.92-1.13). Same kernel, no difference as expected.

2. 4-bit unfused: unsigned consistently faster on x86. AMD median 0.92 (0.85-0.98), Intel median 0.90 (0.79-0.98). Every data point has unsigned <= signed. Effect spans all model sizes. The int4 vs uint4 nibble extraction in the scalar DQ kernel is the likely cause.

3. 8-bit: signed = unsigned on x86. AMD median 0.99, Intel median 0.99. No difference for byte-width types.

4. ARM: median ~1.0 across all three configs. Outliers in both directions, likely measurement variance and memory pressure on large models (same as Sections B and C).

Conclusion: Non-factor for the fused path. The unfused 4-bit DQ kernel runs 8-10% faster with unsigned weights on x86.


---

## E. QDQ Block vs Channel vs Transpose (Weight Layout)

- **Block original**: Fuses to MatMulNBits for 4-bit
- **Block transpose**: Transpose between DQ and MatMul, tests if fusion still works
- **Channel original**: Per-channel quantization, no block structure, cannot fuse
- **Channel transpose**: Per-channel with transposed weights

Note: `channel-sym-signed-transpose` models fail to load across all devices (ORT bug).

In [133]:
# Section E: Effect of weight layout on QDQ performance
# Only 4-bit block_original fuses to MatMulNBits. Everything else runs unfused DQ+MatMul.

qdq_fused = df[(df["format_type"] == "qdq") & (df["scenario"] == "qdq_fused")].copy()
qdq_fused["layout_label"] = qdq_fused["granularity"] + "_" + qdq_fused["weight_layout"]

# Show decode (seq=1) — where layout matters most
for bits in [4, 8]:
    sub = qdq_fused[(qdq_fused["bits"] == bits) & (qdq_fused["symmetry"] == "asym") &
                     (qdq_fused["signedness"] == "signed") & (qdq_fused["seq_length"] == 1)]
    pvt = sub.pivot_table(index=["device", "model_size"], columns="layout_label", values="mean_ms")

    note = "(block_original fuses to MatMulNBits)" if bits == 4 else "(no fusion for any layout)"
    print(f"\n  {bits}-bit decode (seq=1) {note}")
    print("  " + "-" * 90)
    print(pvt.to_string(float_format=lambda x: f"{x:.0f}"))

# Summarize seq=1024 gap (should be much smaller)
sub4_1024 = qdq_fused[(qdq_fused["bits"] == 4) & (qdq_fused["symmetry"] == "asym") &
                       (qdq_fused["signedness"] == "signed") & (qdq_fused["seq_length"] == 1024)]
pvt_1024 = sub4_1024.pivot_table(index=["device", "model_size"], columns="layout_label", values="mean_ms")
ratios = []
for col in [c for c in pvt_1024.columns if c != "block_original"]:
    r = pvt_1024[col] / pvt_1024["block_original"]
    ratios.extend(r.dropna().values)
print(f"\n  4-bit seq=1024: unfused/block_original ratio range = {min(ratios):.1f}-{max(ratios):.1f}x")
print(f"  (Gap shrinks at prefill — compute dominates)")


  4-bit decode (seq=1) (block_original fuses to MatMulNBits)
  ------------------------------------------------------------------------------------------
layout_label       block_original  block_transpose  channel_original  channel_transpose
device model_size                                                                      
amd    0.5b                     8             1013               769                661
       1.5b                    21             3248              2421               2393
       3b                      41             6308              4787               5373
       7b                      79            14857              9618              12249
arm    0.5b                    11              603               618                582
       1.5b                    34             1851              2675               1782
       3b                      56             3626              3678               6880
       7b                      59             9221   

### Observations

1. 4-bit seq=1: block_original is 51-209x faster than all other layouts. Only block_original fuses to MatMulNBits. AMD 80-189x, Intel 143-209x, ARM 51-158x.

2. Block transpose breaks fusion. The Transpose op between DQ and MatMul prevents the QDQ-to-MatMulNBits rewrite. Channel layouts never fuse (no block structure).

3. Among unfused 4-bit layouts (block_transpose, channel_original, channel_transpose), no consistent winner. All run the same slow DQ+MatMul path.

4. 4-bit seq=1024: gap shrinks to 1.0-2.9x. Compute dominates at prefill, DQ overhead is amortized.

5. 8-bit (no layout fuses): block_original is still cheapest, 2-4x faster than other layouts at seq=1. channel_transpose has extreme outliers at large models: AMD 7b 38s, Intel 7b 58s, ARM 3b 36s (15-32x vs block_original). Something pathological in the channel DQ + transpose + MatMul path for large 8-bit models.

6. ARM 7b channel_transpose fails to load for both 4-bit and 8-bit (0 rows). channel-sym-signed-transpose fails to load across all devices 

### E.2 Transpose Overhead Isolation

Section E shows transpose breaks fusion, but the Transpose op adds its own latency too.
Here we isolate the Transpose op cost by comparing unfused-only paths:
block_transpose_unfused vs block_original_unfused (both unfused, only difference is the Transpose op).

This tells us whether model preprocessing to remove Transpose nodes is worthwhile.

In [ ]:
# E.2: Transpose overhead in unfused path
# Compare block-original-unfused vs block-transpose-unfused (both 4-bit unfused)
uf_orig = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) &
    (df["granularity"] == "block") & (df["weight_layout"] == "original") &
    (df["scenario"] == "qdq_unfused") & (df["symmetry"] == "sym") &
    (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(
    columns={"mean_ms": "orig_ms"})

uf_trans = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) &
    (df["granularity"] == "block") & (df["weight_layout"] == "transpose") &
    (df["scenario"].isin(["qdq_unfused", "qdq_fused"])) &
    (df["symmetry"] == "sym") & (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(
    columns={"mean_ms": "trans_ms"})

e2 = uf_orig.merge(uf_trans, on=["device", "model_size", "seq_length"])
e2["transpose_overhead"] = e2["trans_ms"] / e2["orig_ms"]

print("  E.2: Transpose overhead in unfused path (4-bit block sym signed)")
print("  " + "-"*70)
print("  transpose/original ratio (>1.0 means transpose adds latency):")
for seq in [1, 1024]:
    label = "Decode (seq=1)" if seq == 1 else "Prefill (seq=1024)"
    sub = e2[e2["seq_length"] == seq]
    if sub.empty:
        print(f"\n  {label}: no matching data")
        continue
    pivot = sub.pivot_table(index="model_size", columns="device",
                            values="transpose_overhead", aggfunc="first")
    pivot = pivot.reindex(columns=DEVICE_ORDER)
    print(f"\n  {label}:")
    print(pivot.to_string(float_format=lambda x: f"{x:.2f}"))

print("\n  Note: In the fused path, block-original fuses to MatMulNBits while")
print("  block-transpose stays unfused. So the TOTAL impact of transpose is:")
print("  (a) losing fusion (80-150x penalty) + (b) the Transpose op cost shown here.")
print("  Preprocessing to remove Transpose is worthwhile mainly because of (a).")

## F. Unfused DQ Kernel: Block vs Channel

Compare DequantizeLinear kernel performance between **block** and **channel** granularity when both run unfused DQ+MatMul.

**Data sources:**
- **4-bit block**: `qdq_unfused` scenario
- **4-bit channel**: `qdq_fused` scenario — unfused in practice (ORT has no channel-to-MatMulNBits rewrite)
- **8-bit block & channel**: both use `qdq_fused` scenario — neither fuses (the fusion selector only matches 4-bit types)

Filtered to `original` layout (no transpose confound), `asym`, `signed`.

In [156]:
# Unfused DQ Kernel: Block vs Channel
# Both paths run unfused DQ+MatMul — this isolates the DQ kernel cost difference.
# Filtered to original layout, asym, signed.

for bits in [4, 8]:
    # Block unfused: for 4-bit use qdq_unfused scenario; for 8-bit use qdq_fused (doesn't fuse)
    block_scenario = "qdq_unfused" if bits == 4 else "qdq_fused"
    block_unfused = df[
        (df["format_type"] == "qdq") & (df["bits"] == bits) & (df["granularity"] == "block") &
        (df["scenario"] == block_scenario) & (df["weight_layout"] == "original") &
        (df["symmetry"] == "asym") & (df["signedness"] == "signed")
    ][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "block_ms"})

    # Channel unfused: always qdq_fused (never fuses for any bit width)
    channel_unfused = df[
        (df["format_type"] == "qdq") & (df["bits"] == bits) & (df["granularity"] == "channel") &
        (df["scenario"] == "qdq_fused") & (df["weight_layout"] == "original") &
        (df["symmetry"] == "asym") & (df["signedness"] == "signed")
    ][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "channel_ms"})

    combined = block_unfused.merge(channel_unfused, on=["device", "model_size", "seq_length"])
    combined["ch/block"] = combined["channel_ms"] / combined["block_ms"]

    print(f"\n{'='*90}")
    block_note = "qdq_unfused" if bits == 4 else "qdq_fused (unfused in practice)"
    print(f"  Unfused DQ: Block vs Channel — {bits}-bit asym signed, original layout")
    print(f"  Block source: {block_note} | Channel source: qdq_fused (unfused in practice)")
    print(f"  ch/block > 1 = channel slower, < 1 = channel faster")
    print(f"{'='*90}")

    if combined.empty:
        print("  No matching data found.")
        continue

    # Per-device pivot: model_size (rows) × seq_length (cols)
    for dev in DEVICE_ORDER:
        sub = combined[combined["device"] == dev]
        if sub.empty:
            continue
        pvt = sub.pivot_table(index="model_size", columns="seq_length", values="ch/block")
        pvt = pvt.reindex(MODEL_SIZE_ORDER).dropna(how="all")
        print(f"\n  {dev.upper()} — ch/block ratio")
        print("  " + "-"*60)
        print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))

    # Absolute latencies side by side for seq=1 (decode)
    seq1_dq = combined[combined["seq_length"] == 1].set_index(["device", "model_size"]).sort_index()
    if not seq1_dq.empty:
        print(f"\n  Absolute latency at seq=1 (ms):")
        print("  " + "-"*60)
        print(seq1_dq[["block_ms", "channel_ms", "ch/block"]].to_string(float_format=lambda x: f"{x:.1f}"))

    # Cross-device summary
    print(f"\n  Per-device summary ({bits}-bit):")
    for dev in DEVICE_ORDER:
        ratios = combined[combined["device"] == dev]["ch/block"]
        if ratios.empty:
            continue
        print(f"    {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  "
              f"median={ratios.median():.2f}  mean={ratios.mean():.2f}")



  Unfused DQ: Block vs Channel — 4-bit asym signed, original layout
  Block source: qdq_unfused | Channel source: qdq_fused (unfused in practice)
  ch/block > 1 = channel slower, < 1 = channel faster

  AMD — ch/block ratio
  ------------------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        1.11  0.98  0.95  0.79  0.72
1.5b        1.07  0.97  0.94  0.84  0.77
3b          1.09  0.96  0.87  0.80  0.79
7b          1.18  1.03  0.94  0.85  0.82

  INTEL — ch/block ratio
  ------------------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        1.58  1.37  1.34  1.24  1.14
1.5b        1.43  1.31  1.26  1.21  1.13
3b          1.45  1.24  1.19  1.13  1.04
7b          1.42  1.30  1.30  1.21  1.14

  ARM — ch/block ratio
  ------------------------------------------------------------
seq_length  1     128   256   512   1024
mod

### Observations

**4-bit block vs channel**

1. AMD: channel 7-18% slower at seq=1, crosses over around seq=256 and becomes faster (0.72-0.95x). Median 0.94x. Consistent across all model sizes.

2. Intel: channel 42-58% slower at seq=1, narrows to 4-14% at seq=1024. Median 1.25x. Channel always slower, gap just shrinks.

3. ARM: 1.5b is an outlier (1.49x at seq=1, 1.35x at seq=128, falls to 0.95x at seq=1024). 0.5b and 3b near parity (0.88-1.06x). 7b slightly favors channel across all seq lengths (0.83-0.95x).

**8-bit block vs channel**

4. AMD: channel 2.9-3.4x slower at seq=1. Drops below 1.0x at seq=1024 (0.80-0.94x). Same crossover pattern as 4-bit but starting gap much larger.

5. Intel: channel 2.8-5.1x slower at seq=1. Still 1.18-1.38x at seq=1024. Never reaches parity.

6. ARM 0.5b-3b: channel 3.5-7.0x slower at seq=1. 3b seq=1024 drops to 0.43x (sudden jump from 1.33x at seq=512, suspect measurement). 7b inverted at all seq lengths (0.37-0.64x), model exceeds device RAM.

Conclusion: Block DQ is faster than channel DQ at decode. 4-bit gap is moderate (7-18% on AMD, up to 49% on ARM 1.5b, up to 58% on Intel). 8-bit gap is large (3-7x at decode). Gap shrinks at prefill on all devices. AMD crosses over (channel faster at long seq). Intel channel never fully catches up even at seq=1024.

## G. Channel DQ: 4-bit vs 8-bit

Both 4-bit and 8-bit channel run unfused DQ+MatMul (no fusion possible). The DQ kernel is the only difference. 4-bit needs bit extraction, 8-bit reads full bytes.

In [157]:
# E.3: Channel DQ 4-bit vs 8-bit (both unfused)
# Both use qdq_fused scenario but neither actually fuses.
# Filtered to channel, original layout, asym, signed.

for dev in DEVICE_ORDER:
    ch4 = df[(df["format_type"]=="qdq") & (df["bits"]==4) & (df["granularity"]=="channel") &
             (df["scenario"]=="qdq_fused") & (df["weight_layout"]=="original") &
             (df["symmetry"]=="asym") & (df["signedness"]=="signed") &
             (df["device"]==dev)][["model_size","seq_length","mean_ms"]].rename(columns={"mean_ms":"ch4_ms"})
    ch8 = df[(df["format_type"]=="qdq") & (df["bits"]==8) & (df["granularity"]=="channel") &
             (df["scenario"]=="qdq_fused") & (df["weight_layout"]=="original") &
             (df["symmetry"]=="asym") & (df["signedness"]=="signed") &
             (df["device"]==dev)][["model_size","seq_length","mean_ms"]].rename(columns={"mean_ms":"ch8_ms"})
    m = ch4.merge(ch8, on=["model_size","seq_length"])
    m["8b/4b"] = m["ch8_ms"] / m["ch4_ms"]
    pvt = m.pivot_table(index="model_size", columns="seq_length", values="8b/4b")
    pvt = pvt.reindex(MODEL_SIZE_ORDER).dropna(how="all")
    print(f"\n  {dev.upper()} — channel 8b/4b ratio (< 1 = 8-bit faster)")
    print("  " + "-"*50)
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))

    # seq=1 absolute values
    s1 = m[m["seq_length"]==1].sort_values("model_size")
    print(f"  seq=1 absolute (ms):")
    for _, row in s1.iterrows():
        print(f"    {row['model_size']:4s}  4b={row['ch4_ms']:.0f}  8b={row['ch8_ms']:.0f}  8b/4b={row['8b/4b']:.2f}")


  AMD — channel 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.82  0.85  0.75  0.94  1.03
1.5b        0.81  0.84  0.89  0.91  0.97
3b          0.78  0.82  0.85  0.87  0.84
7b          0.78  0.89  0.99  1.00  0.97
  seq=1 absolute (ms):
    0.5b  4b=769  8b=628  8b/4b=0.82
    1.5b  4b=2421  8b=1965  8b/4b=0.81
    3b    4b=4787  8b=3742  8b/4b=0.78
    7b    4b=9618  8b=7455  8b/4b=0.78

  INTEL — channel 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.68  0.73  0.80  0.86  0.90
1.5b        0.72  0.76  0.82  0.84  0.90
3b          0.52  0.64  0.71  0.80  0.87
7b          0.70  0.78  0.82  0.84  0.89
  seq=1 absolute (ms):
    0.5b  4b=1552  8b=1059  8b/4b=0.68
    1.5b  4b=4565  8b=3270  8b/4b=0.72
    3b    4b=9044  8b=

### Observations

1. x86 seq=1: 8-bit channel is faster than 4-bit channel everywhere. AMD 8b/4b = 0.78-0.82x, Intel 0.52-0.72x. Both paths run unfused DQ to fp32 then the same MatMul, so the gap is purely DQ kernel cost. Nibble extraction in the 4-bit DQ kernel costs more than the 2x smaller input saves.

2. Intel 3b is an outlier at seq=1: 0.52x (4b=9044ms, 8b=4695ms). Other Intel models are 0.68-0.72x. Not clear why 3b benefits disproportionately.

3. x86 at higher seq: gap narrows. Intel stays 8-bit-faster at all seq lengths (0.87-0.90x at seq=1024). AMD is mixed at seq=1024: 0.5b crosses 1.0 (4-bit becomes slightly faster at 1.03x), 3b still 0.84, rest ~0.97.

4. ARM: 8-bit is slower for 3 of 4 models (0.5b 1.14-1.45x, 3b 1.05-1.09x, 7b 1.12-1.27x). 1.5b is the exception (0.79-1.02x, 8-bit faster). ARM's nibble extraction appears cheaper than x86's, so the 2x bandwidth increase from 8-bit is not offset by simpler unpacking.

Conclusion: On x86, unfused 8-bit channel DQ is 18-48% faster than 4-bit channel DQ at decode. Nibble extraction overhead outweighs the bandwidth advantage on x86. On ARM, opposite direction: 8-bit is slower for most models, suggesting ARM's nibble unpacking is cheaper.

## H. Unfused DQ: 4-bit vs 8-bit



In [ ]:
# Section H: Unfused 4-bit vs 8-bit — both run DQ+MatMul
# 4-bit: qdq_unfused scenario (explicitly unfused)
# 8-bit: qdq_fused scenario (doesn't actually fuse — fusion selector only matches 4-bit)
qdq_4_unfused = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_unfused") & (df["weight_layout"] == "original") &
    (df["symmetry"] == "asym") & (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "4bit_ms"})

qdq_8_unfused = df[
    (df["format_type"] == "qdq") & (df["bits"] == 8) & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_fused") & (df["weight_layout"] == "original") &
    (df["symmetry"] == "asym") & (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "8bit_ms"})

combined = qdq_4_unfused.merge(qdq_8_unfused, on=["device", "model_size", "seq_length"])
combined["8b/4b"] = combined["8bit_ms"] / combined["4bit_ms"]

for dev in DEVICE_ORDER:
    dev_df = combined[combined["device"] == dev]
    pvt = dev_df.pivot_table(index="model_size", columns="seq_length", values="8b/4b")
    pvt = pvt.reindex(MODEL_SIZE_ORDER).dropna(how="all")
    print(f"\n  {dev.upper()} — unfused 8b/4b ratio (< 1 = 8-bit faster)")
    print("  " + "-"*50)
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))

    # Show absolute values at seq=1
    s1 = dev_df[dev_df["seq_length"] == 1].sort_values("model_size")
    print(f"  seq=1 absolute (ms):")
    for _, row in s1.iterrows():
        print(f"    {row['model_size']:4s}  4b={row['4bit_ms']:.0f}  8b={row['8bit_ms']:.0f}  8b/4b={row['8b/4b']:.2f}")

    # Show absolute values at seq=1024
    s1024 = dev_df[dev_df["seq_length"] == 1024].sort_values("model_size")
    print(f"  seq=1024 absolute (ms):")
    for _, row in s1024.iterrows():
        print(f"    {row['model_size']:4s}  4b={row['4bit_ms']:.0f}  8b={row['8bit_ms']:.0f}  8b/4b={row['8b/4b']:.2f}")


  AMD — 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.27  0.45  0.55  0.69  0.82
1.5b        0.26  0.43  0.55  0.70  0.81
3b          0.28  0.45  0.57  0.71  0.83
7b          0.31  0.50  0.62  0.74  0.84

  INTEL — 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.21  0.38  0.50  0.68  0.79
1.5b        0.22  0.35  0.46  0.61  0.73
3b          0.26  0.39  0.50  0.65  0.77
7b          0.24  0.37  0.49  0.64  0.77

  ARM — 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.21  0.43  0.61  0.72  0.91
1.5b        0.31  0.50  0.61  0.76  0.92
3b          0.33  0.53  0.62  0.82  2.21
7b          1.94

### Observations

**AMD:** 8-bit is 3-4x faster at seq=1 (ratio 0.26-0.31). Gap narrows steadily to 0.81-0.84 at seq=1024. Model size has minimal effect. Clean, predictable scaling.

**Intel:** 8-bit is 4-5x faster at seq=1 (ratio 0.21-0.26). Larger gap than AMD at every seq_length. Narrows to 0.73-0.79 at seq=1024. Model size has minimal effect. Same clean scaling as AMD.

**ARM 0.5b-1.5b:** Follows x86 pattern. seq=1: 0.21-0.31. Slightly higher ratios at seq=1024 (0.91-0.92 vs x86's 0.73-0.84) meaning 4-bit DQ is relatively cheaper on ARM than x86 at prefill.

**ARM 3b:** Normal through seq=512 (0.33-0.82). Breaks at seq=1024: ratio jumps to 2.21 (8-bit is 2.2x SLOWER). 8-bit model is ~2x larger, pushing past comfortable RAM on 16GB.

**ARM 7b:** 8-bit slower at every seq_length (1.66-2.82x). Worst at seq=128 (2.82x). Severe memory pressure. Same issue as Sections C and D.

**Cross-device:** AMD and Intel agree within 0.1 everywhere. ARM diverges only at 3b/seq=1024 and all of 7b (spread > 0.3). Root cause is memory, not kernel differences.

**TLDR:** 8-bit unfused is always faster on x86 (3-5x at decode, 1.2-1.4x at prefill). Same on ARM for models that fit in RAM. On memory-constrained ARM, larger 8-bit model size negates the faster kernel.

## I. Model Load Time

Session creation time across formats/layouts/devices. Includes graph optimization + weight preparation.

In [167]:
# Section I: Model load time
# One load time per model (not per seq_length), so deduplicate
load_times = df.drop_duplicates(
    subset=["device", "format_type", "model_size", "bits", "symmetry",
            "granularity", "signedness", "weight_layout", "scenario"]
)[["device", "format_type", "model_size", "bits", "symmetry", "granularity",
   "signedness", "weight_layout", "scenario", "model_load_time_s"]].copy()

load_times = load_times.sort_values(["device", "format_type", "weight_layout", "bits", "model_size"])

# I.1: MNB load times
mnb_load = load_times[load_times["format_type"] == "mnb"]
tbl = mnb_load.pivot_table(
    index=["bits", "symmetry", "model_size"],
    columns="device",
    values="model_load_time_s"
)
print(f"\n{'='*80}")
print(f"  I.1 MNB model load times (seconds)")
print(f"{'='*80}")
print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

# I.2: QDQ load times — focus on block fused, original vs transpose
qdq_load = load_times[
    (load_times["format_type"] == "qdq") & (load_times["granularity"] == "block") &
    (load_times["scenario"] == "qdq_fused") & (load_times["signedness"] == "signed") &
    (load_times["symmetry"] == "asym")
]
tbl2 = qdq_load.pivot_table(
    index=["bits", "weight_layout", "model_size"],
    columns="device",
    values="model_load_time_s"
)
print(f"\n{'='*80}")
print(f"  I.2 QDQ block asym signed fused — load times (seconds)")
print(f"{'='*80}")
print(tbl2.to_string(float_format=lambda x: f"{x:.2f}"))

# I.3: Flag extreme load times (>30s)
extreme = load_times[load_times["model_load_time_s"] > 30]
if not extreme.empty:
    print(f"\n{'='*80}")
    print(f"  I.3 EXTREME load times (>30s) — possible issues")
    print(f"{'='*80}")
    print(extreme.to_string(index=False, float_format=lambda x: f"{x:.1f}"))
else:
    print("\nNo extreme load times (>30s) detected.")



  I.1 MNB model load times (seconds)
device                     amd   arm  intel
bits symmetry model_size                   
4    asym     0.5b        0.78  0.49   1.32
              1.5b        1.83  3.58   2.25
              3b          5.90  6.46   4.56
              7b          9.97 12.76   9.76
     sym      0.5b        0.85  1.85   1.14
              1.5b        2.15  4.56   2.86
              3b          6.86  8.05   5.70
              7b          9.47 18.42  12.35
8    asym     0.5b        0.87  3.15   1.17
              1.5b        3.84  6.58   2.82
              3b          6.85 12.17   6.01
              7b         12.64 48.39  13.48
     sym      0.5b        0.97  2.84   1.27
              1.5b        4.20  8.07   3.43
              3b          7.40 13.65   6.96
              7b         13.28 51.37  15.62

  I.2 QDQ block asym signed fused — load times (seconds)
device                          amd   arm  intel
bits weight_layout model_size                   
4    original 

In [170]:
# I.4: QDQ unfused load times — original vs transpose (4-bit and 8-bit)
# Both unfused paths skip PrePack (no fusion), so load times should be fast.
# This isolates whether the Transpose node in the graph adds load-time overhead.
# Note: 8-bit QDQ never fuses, so qdq_fused scenario is effectively unfused.

for bits in [4, 8]:
    if bits == 4:
        unfused_load = load_times[
            (load_times["format_type"] == "qdq") & (load_times["bits"] == 4) &
            (load_times["granularity"] == "block") & (load_times["scenario"] == "qdq_unfused") &
            (load_times["signedness"] == "signed")
        ]
        scenario_note = "qdq_unfused scenario"
    else:
        # 8-bit: use qdq_fused scenario (doesn't actually fuse — no PrePack)
        unfused_load = load_times[
            (load_times["format_type"] == "qdq") & (load_times["bits"] == 8) &
            (load_times["granularity"] == "block") & (load_times["scenario"] == "qdq_fused") &
            (load_times["signedness"] == "signed")
        ]
        scenario_note = "qdq_fused scenario (unfused in practice — 8-bit never fuses)"

    tbl_unfused = unfused_load.pivot_table(
        index=["symmetry", "weight_layout", "model_size"],
        columns="device",
        values="model_load_time_s"
    )

    print(f"\n{'='*80}")
    print(f"  I.4 QDQ {bits}-bit block signed — load times (seconds)")
    print(f"  Source: {scenario_note}")
    print(f"  No PrePack triggered. Transpose = extra graph node only.")
    print(f"{'='*80}")
    print(tbl_unfused.to_string(float_format=lambda x: f"{x:.3f}"))

    # Side-by-side original vs transpose with ratio
    for sym in ["asym", "sym"]:
        orig = unfused_load[
            (unfused_load["symmetry"] == sym) & (unfused_load["weight_layout"] == "original")
        ][["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "original_s"})
        trans = unfused_load[
            (unfused_load["symmetry"] == sym) & (unfused_load["weight_layout"] == "transpose")
        ][["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "transpose_s"})
        merged = orig.merge(trans, on=["device", "model_size"])
        if merged.empty:
            continue
        merged["trans/orig"] = merged["transpose_s"] / merged["original_s"]
        merged = merged.set_index(["device", "model_size"]).sort_index()
        print(f"\n  {bits}-bit {sym} — original vs transpose:")
        print("  " + "-"*60)
        print(merged.to_string(float_format=lambda x: f"{x:.3f}"))

# I.5: Three-way comparison (4-bit only — only 4-bit fused triggers PrePack)
print(f"\n{'='*80}")
print(f"  I.5 Load time comparison: fused original vs unfused original vs unfused transpose")
print(f"  (4-bit block asym signed — shows PrePack cost)")
print(f"{'='*80}")

qdq_unfused_4bit_load = load_times[
    (load_times["format_type"] == "qdq") & (load_times["bits"] == 4) &
    (load_times["granularity"] == "block") & (load_times["scenario"] == "qdq_unfused") &
    (load_times["signedness"] == "signed")
]

fused_orig = load_times[
    (load_times["format_type"] == "qdq") & (load_times["bits"] == 4) &
    (load_times["granularity"] == "block") & (load_times["scenario"] == "qdq_fused") &
    (load_times["weight_layout"] == "original") & (load_times["signedness"] == "signed") &
    (load_times["symmetry"] == "asym")
][["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "fused_orig_s"})

unfused_orig = qdq_unfused_4bit_load[
    (qdq_unfused_4bit_load["symmetry"] == "asym") & (qdq_unfused_4bit_load["weight_layout"] == "original")
][["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "unfused_orig_s"})

unfused_trans = qdq_unfused_4bit_load[
    (qdq_unfused_4bit_load["symmetry"] == "asym") & (qdq_unfused_4bit_load["weight_layout"] == "transpose")
][["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "unfused_trans_s"})

three_way = fused_orig.merge(unfused_orig, on=["device", "model_size"]).merge(unfused_trans, on=["device", "model_size"])
three_way["prepack_cost_s"] = three_way["fused_orig_s"] - three_way["unfused_orig_s"]
three_way = three_way.set_index(["device", "model_size"]).sort_index()
print(three_way.to_string(float_format=lambda x: f"{x:.3f}"))


  I.4 QDQ 4-bit block signed — load times (seconds)
  Source: qdq_unfused scenario
  No PrePack triggered. Transpose = extra graph node only.
device                              amd   arm  intel
symmetry weight_layout model_size                   
asym     original      0.5b       0.151 0.161  0.192
                       1.5b       0.155 0.265  0.208
                       3b         0.209 0.324  0.382
                       7b         0.161 0.239  0.314
         transpose     0.5b       0.171   NaN  0.235
                       1.5b       0.276   NaN  0.252
                       3b         0.287   NaN  0.385
                       7b         0.271   NaN  0.356
sym      original      0.5b       0.101 0.201  0.168
                       1.5b       0.114 0.241  0.273
                       3b         0.142 0.283  0.226
                       7b         0.119 0.172  0.274
         transpose     0.5b       0.128   NaN  0.180
                       1.5b       0.168   NaN  0.225
         

In [173]:
# I.6  Fused transpose vs fused original — 4-bit load time
# Fused original: fusion succeeds → MatMulNBits PrePack (weight repacking at load)
# Fused transpose: Transpose node blocks fusion → no PrePack, fast load

print("I.6  Fused 4-bit load time: transpose vs original (block, signed)")
print("=" * 70)

fl = load_times[
    (load_times["format_type"] == "qdq") & (load_times["bits"] == 4) &
    (load_times["granularity"] == "block") & (load_times["scenario"] == "qdq_fused") &
    (load_times["signedness"] == "signed")
]
fo = fl[fl["weight_layout"] == "original"][["device","model_size","symmetry","model_load_time_s"]].rename(columns={"model_load_time_s":"fused_orig_s"})
ft = fl[fl["weight_layout"] == "transpose"][["device","model_size","symmetry","model_load_time_s"]].rename(columns={"model_load_time_s":"fused_trans_s"})
fm = fo.merge(ft, on=["device","model_size","symmetry"])
fm["ratio"] = fm["fused_trans_s"] / fm["fused_orig_s"]

for dev in ["amd", "intel", "arm"]:
    sub = fm[fm["device"] == dev].sort_values(["model_size","symmetry"])
    if sub.empty:
        print(f"\n{dev}: no transpose data")
        continue
    print(f"\n{dev}:")
    print(f"  {'model':<6} {'sym':<5} {'fused_orig(s)':>13} {'fused_trans(s)':>14} {'trans/orig':>10}")
    print(f"  {'-'*6} {'-'*5} {'-'*13} {'-'*14} {'-'*10}")
    for _, row in sub.iterrows():
        print(f"  {row['model_size']:<6} {row['symmetry']:<5} {row['fused_orig_s']:>13.2f} {row['fused_trans_s']:>14.2f} {row['ratio']:>10.3f}")
    print(f"  median ratio: {sub['ratio'].median():.3f}x")


I.6  Fused 4-bit load time: transpose vs original (block, signed)

amd:
  model  sym   fused_orig(s) fused_trans(s) trans/orig
  ------ ----- ------------- -------------- ----------
  0.5b   asym           1.08           0.20      0.182
  0.5b   sym            1.12           0.18      0.158
  1.5b   asym           2.73           0.28      0.102
  1.5b   sym            3.04           0.22      0.072
  3b     asym           5.81           0.34      0.058
  3b     sym            6.08           0.29      0.047
  7b     asym          12.80           0.32      0.025
  7b     sym           13.21           0.24      0.018
  median ratio: 0.065x

intel:
  model  sym   fused_orig(s) fused_trans(s) trans/orig
  ------ ----- ------------- -------------- ----------
  0.5b   asym           1.21           0.43      0.359
  0.5b   sym            2.09           0.23      0.112
  1.5b   asym           3.27           0.37      0.114
  1.5b   sym            3.73           0.36      0.096
  3b     asym    

### Observations

1. Load time is driven by PrePack (weight transformation). Only MNB and 4-bit QDQ original trigger it. All other QDQ paths (4-bit transpose, all 8-bit) load in ~0.5s or less regardless of model size.

2. 4-bit QDQ fused vs MNB: 0.9-1.5x on x86, 1.3-1.6x on ARM (0.5b ARM outlier at 3.4x from fixed graph-rewrite cost on a small model). AMD 3b and Intel 0.5b load slightly faster than MNB. Also visible in Section A Table 2.

3. MNB 8-bit vs QDQ 8-bit load: 0.9-51s vs 0.16-0.51s. MNB does PrePack at load, QDQ 8-bit does not fuse so no PrePack. Same reason QDQ unfused loads fast in Section A.

4. Channel transpose (40-228s) and ARM 7b 8-bit memory (48-51s): already covered in Sections E and C.

## J Data Validation

In [160]:
# Section J: Data validation

print("="*80)
print("  J.1 Row counts per device")
print("="*80)
print(df.groupby("device").size().to_string())

print(f"\n{'='*80}")
print("  J.2 Breakdown by device × format_type × weight_layout × scenario")
print("="*80)
breakdown = df.groupby(["device", "format_type", "weight_layout", "scenario"]).size().unstack(fill_value=0)
print(breakdown.to_string())

print(f"\n{'='*80}")
print("  J.3 Configs per device (unique format+size+bits+sym+gran+sign+layout+scenario)")
print("="*80)
for device in DEVICE_ORDER:
    dev_df = df[df["device"] == device]
    configs = dev_df.drop_duplicates(
        subset=["format_type", "model_size", "bits", "symmetry", "granularity",
                "signedness", "weight_layout", "scenario"]
    )
    print(f"  {device}: {len(configs)} unique configs, {len(dev_df)} total rows")

# J.4: Check for duplicates after disambiguation
print(f"\n{'='*80}")
print("  J.4 Duplicate check (same device+config+seq_length)")
print("="*80)
group_cols = ["device", "format_type", "model_size", "bits", "symmetry",
              "granularity", "signedness", "weight_layout", "scenario", "seq_length"]
dupes = df.groupby(group_cols).size()
dupes_gt1 = dupes[dupes > 1]
if dupes_gt1.empty:
    print("  No duplicates found — data is clean!")
else:
    print(f"  {len(dupes_gt1)} duplicate groups found:")
    print(dupes_gt1.to_string())

# J.5: Missing configs across devices
print(f"\n{'='*80}")
print("  J.5 Configs present in one device but missing from another (seq=1 only)")
print("="*80)
config_cols = ["format_type", "model_size", "bits", "symmetry", "granularity",
               "signedness", "weight_layout", "scenario"]
decode_df = df[df["seq_length"] == 1]
for d1 in DEVICE_ORDER:
    for d2 in DEVICE_ORDER:
        if d1 >= d2:
            continue
        s1 = set(decode_df[decode_df["device"] == d1][config_cols].apply(tuple, axis=1))
        s2 = set(decode_df[decode_df["device"] == d2][config_cols].apply(tuple, axis=1))
        only_d1 = s1 - s2
        only_d2 = s2 - s1
        if only_d1 or only_d2:
            print(f"\n  {d1} vs {d2}:")
            if only_d1:
                print(f"    Only in {d1} ({len(only_d1)}):", list(only_d1)[:5], "..." if len(only_d1) > 5 else "")
            if only_d2:
                print(f"    Only in {d2} ({len(only_d2)}):", list(only_d2)[:5], "..." if len(only_d2) > 5 else "")


  J.1 Row counts per device
device
amd      640
arm      560
intel    640

  J.2 Breakdown by device × format_type × weight_layout × scenario
scenario                          native  qdq_fused  qdq_unfused
device format_type weight_layout                                
amd    mnb         original           80          0            0
       qdq         original            0        240           60
                   transpose           0        200           60
arm    mnb         original           80          0            0
       qdq         original            0        240           60
                   transpose           0        180            0
intel  mnb         original           80          0            0
       qdq         original            0        240           60
                   transpose           0        200           60

  J.3 Configs per device (unique format+size+bits+sym+gran+sign+layout+scenario)
  amd: 128 unique configs, 640 total rows
  intel: 128 unique

## K. Latency Scaling with Model Size

Check if latency scales linearly with parameter count. For bandwidth-bound decode (seq=1),
it should. Deviations indicate cache effects, NUMA issues, or memory pressure.

Parameter counts: 0.5b=0.5, 1.5b=1.5, 3b=3, 7b=7. Ratios relative to 0.5b: 3x, 6x, 14x.

In [ ]:
# K: Latency scaling linearity
PARAM_COUNTS = {"0.5b": 0.5, "1.5b": 1.5, "3b": 3.0, "7b": 7.0}
PARAM_RATIOS = {ms: pc / 0.5 for ms, pc in PARAM_COUNTS.items()}

print("  K. Latency Scaling \u2014 decode (seq=1) relative to 0.5b")
print("  Expected ratios: 0.5b=1.0x  1.5b=3.0x  3b=6.0x  7b=14.0x")
print("  " + "="*80)

configs_to_check = [
    ("MNB 4-bit sym", df[(df["format_type"]=="mnb") & (df["bits"]==4) & (df["symmetry"]=="sym") & (df["weight_layout"]=="original")]),
    ("MNB 8-bit sym", df[(df["format_type"]=="mnb") & (df["bits"]==8) & (df["symmetry"]=="sym") & (df["weight_layout"]=="original")]),
    ("QDQ 4-bit unfused", df[(df["format_type"]=="qdq") & (df["bits"]==4) & (df["granularity"]=="block") & (df["scenario"]=="qdq_unfused") & (df["weight_layout"]=="original") & (df["symmetry"]=="asym") & (df["signedness"]=="signed")]),
]

for label, data in configs_to_check:
    print(f"\n  {label}:")
    for dev in DEVICE_ORDER:
        seq1 = data[(data["device"]==dev) & (data["seq_length"]==1)].set_index("model_size")["mean_ms"]
        if seq1.empty or "0.5b" not in seq1.index:
            continue
        base = seq1["0.5b"]
        actual = {ms: seq1[ms]/base for ms in MODEL_SIZE_ORDER if ms in seq1.index}
        dev_r = {ms: actual[ms]/PARAM_RATIOS[ms] for ms in actual}
        r_str = "  ".join(f"{ms}={actual[ms]:.1f}x" for ms in actual)
        d_str = "  ".join(f"{ms}={dev_r[ms]:.2f}" for ms in dev_r)
        print(f"    {dev:6s} actual: {r_str}")
        print(f"    {' ':6s} actual/expected: {d_str}  (1.0 = perfect linear)")

## L. Decode vs Prefill Crossover Analysis

At what sequence length does the unfused overhead drop below key thresholds?
This is the most actionable number for users deciding whether QDQ overhead matters for their workload.

In [ ]:
# L: Crossover analysis
fused_4_l = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_fused") & (df["weight_layout"] == "original") &
    (df["symmetry"] == "asym") & (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "fused_ms"})

unfused_4_l = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_unfused") & (df["weight_layout"] == "original") &
    (df["symmetry"] == "asym") & (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "unfused_ms"})

cross = fused_4_l.merge(unfused_4_l, on=["device", "model_size", "seq_length"])
cross["ratio"] = cross["unfused_ms"] / cross["fused_ms"]

print("  L. Unfused/Fused ratio by seq_length")
print("  " + "="*80)

for dev in DEVICE_ORDER:
    pvt = cross[cross["device"]==dev].pivot_table(index="model_size", columns="seq_length", values="ratio")
    pvt = pvt.reindex(MODEL_SIZE_ORDER).dropna(how="all")
    print(f"\n  {dev.upper()}:")
    print(pvt.to_string(float_format=lambda x: f"{x:.1f}"))

print(f"\n  Estimated seq_length where unfused/fused drops below thresholds:")
print(f"  {'device':<8} {'model':<6} {'<5x':>8} {'<2x':>8} {'<1.5x':>8}")
print(f"  {'-'*8} {'-'*6} {'-'*8} {'-'*8} {'-'*8}")
for dev in DEVICE_ORDER:
    for ms in MODEL_SIZE_ORDER:
        sub = cross[(cross["device"]==dev) & (cross["model_size"]==ms)].sort_values("seq_length")
        if sub.empty: continue
        results = []
        for thresh in [5, 2, 1.5]:
            below = sub[sub["ratio"] <= thresh]
            if below.empty: results.append(">1024")
            elif sub[sub["ratio"] > thresh].empty: results.append("<1")
            else:
                fsq = below["seq_length"].iloc[0]
                above = sub[sub["seq_length"] < fsq]
                if above.empty: results.append(f"<{fsq}")
                else:
                    s1, r1 = above.iloc[-1]["seq_length"], above.iloc[-1]["ratio"]
                    s2, r2 = fsq, below.iloc[0]["ratio"]
                    est = s1 + (thresh - r1) * (s2 - s1) / (r2 - r1) if r1 != r2 else s1
                    results.append(f"~{int(est)}")
        print(f"  {dev:<8} {ms:<6} {results[0]:>8} {results[1]:>8} {results[2]:>8}")

## M. Cross-Device Performance Analysis

Compare the same configuration across devices to surface device-specific strengths, weaknesses, and anomalies.

In [ ]:
# M: Cross-device comparison
print("  M. Cross-Device Performance")
print("  " + "="*80)

print("\n  M.1 MNB 4-bit sym decode (seq=1) relative to AMD:")
print("  " + "-"*60)
mnb4s_m = df[(df["format_type"]=="mnb") & (df["bits"]==4) & (df["symmetry"]=="sym") & (df["weight_layout"]=="original") & (df["seq_length"]==1)]
pvt_m = mnb4s_m.pivot_table(index="model_size", columns="device", values="mean_ms")
pvt_m = pvt_m.reindex(MODEL_SIZE_ORDER).dropna(how="all")
print("  Absolute (ms):")
print(pvt_m.to_string(float_format=lambda x: f"{x:.1f}"))
pvt_norm = pvt_m.div(pvt_m["amd"], axis=0)
print("\n  Relative to AMD:")
print(pvt_norm.to_string(float_format=lambda x: f"{x:.2f}"))

print(f"\n  M.2 Unfused 4-bit DQ penalty at seq=1:")
print("  " + "-"*60)
mnb_4m = df[(df["format_type"]=="mnb") & (df["bits"]==4) & (df["symmetry"]=="asym") & (df["weight_layout"]=="original") & (df["seq_length"]==1)]
qdq_uf_m = df[(df["format_type"]=="qdq") & (df["bits"]==4) & (df["granularity"]=="block") & (df["scenario"]=="qdq_unfused") & (df["weight_layout"]=="original") & (df["symmetry"]=="asym") & (df["signedness"]=="signed") & (df["seq_length"]==1)]
mm = mnb_4m[["device","model_size","mean_ms"]].rename(columns={"mean_ms":"mnb_ms"}).merge(
    qdq_uf_m[["device","model_size","mean_ms"]].rename(columns={"mean_ms":"unfused_ms"}), on=["device","model_size"])
mm["unfused/mnb"] = mm["unfused_ms"] / mm["mnb_ms"]
pvt_gap = mm.pivot_table(index="model_size", columns="device", values="unfused/mnb")
pvt_gap = pvt_gap.reindex(MODEL_SIZE_ORDER).dropna(how="all")
print("  Unfused/MNB ratio (higher = worse penalty):")
print(pvt_gap.to_string(float_format=lambda x: f"{x:.0f}"))
print(f"\n  Intel consistently shows the highest unfused penalty.")

print(f"\n  M.3 ARM-Specific Anomalies:")
print("  " + "-"*60)
print("  a) ARM asym penalty (MNB 4-bit, seq=1):")
arm_4m = df[(df["device"]=="arm") & (df["format_type"]=="mnb") & (df["bits"]==4) & (df["weight_layout"]=="original") & (df["seq_length"]==1)]
arm_pvt = arm_4m.pivot_table(index="model_size", columns="symmetry", values="mean_ms")
arm_pvt = arm_pvt.reindex(MODEL_SIZE_ORDER).dropna(how="all")
arm_pvt["asym/sym"] = arm_pvt["asym"] / arm_pvt["sym"]
print(arm_pvt.to_string(float_format=lambda x: f"{x:.1f}"))

print("\n  b) ARM 8-bit faster than 4-bit at seq>=128 (opposite of x86):")
arm_mnb_m = df[(df["device"]=="arm") & (df["format_type"]=="mnb") & (df["symmetry"]=="sym") & (df["weight_layout"]=="original") & (df["seq_length"]==128)]
arm_bits = arm_mnb_m.pivot_table(index="model_size", columns="bits", values="mean_ms")
arm_bits = arm_bits.reindex(MODEL_SIZE_ORDER).dropna(how="all")
arm_bits["8b/4b"] = arm_bits[8] / arm_bits[4]
print(arm_bits.to_string(float_format=lambda x: f"{x:.1f}"))
print("  8b/4b < 1.0 = 8-bit is faster. ARM nibble unpacking is expensive.")
print("\n  c) ARM 7b 8-bit: model exceeds 16GB device RAM. Unusable.")

### M. Observations

**Intel** has the worst unfused DQ penalty despite similar MNB performance to AMD. The scalar DQ loop
may be hitting Intel's weaker single-thread scalar throughput or less favorable cache behavior.
Optimization priority: Intel benefits most from threading/SIMD in the DQ kernel.

**ARM** has three unique issues:
1. **Asym penalty**: 4-bit MNB asym is ~50% slower than sym for models >=1.5b. Not seen on x86.
   Likely an ARM kernel zero-point handling issue that deserves investigation.
2. **8-bit faster than 4-bit at prefill**: ARM's nibble unpacking is expensive enough that
   reading 2x more bytes (8-bit) is still faster than extracting nibbles (4-bit).
3. **7b 8-bit unusable**: Exceeds 16GB device RAM, causes severe memory pressure.

**AMD** is the most predictable device with clean scaling and no major anomalies.

## N. Effective Memory Bandwidth Estimation

At decode (seq=1), inference is bandwidth-bound: we read all weights once and do minimal compute.
We can estimate effective memory bandwidth from latency and weight size to see how close each
path gets to theoretical peak.

In [ ]:
# N: Bandwidth estimation
WEIGHT_SIZES_GB = {
    ("0.5b", 4): 0.5 * 0.5e9 / 1e9 * 1.08,
    ("1.5b", 4): 1.5 * 0.5e9 / 1e9 * 1.08,
    ("3b", 4): 3.0 * 0.5e9 / 1e9 * 1.08,
    ("7b", 4): 7.0 * 0.5e9 / 1e9 * 1.08,
    ("0.5b", 8): 0.5 * 1.0e9 / 1e9 * 1.05,
    ("1.5b", 8): 1.5 * 1.0e9 / 1e9 * 1.05,
    ("3b", 8): 3.0 * 1.0e9 / 1e9 * 1.05,
    ("7b", 8): 7.0 * 1.0e9 / 1e9 * 1.05,
}

print("  N. Effective Memory Bandwidth (GB/s) at decode (seq=1)")
print("  " + "="*80)
print("  Bandwidth = weight_size / latency. Higher = better.")
print("  Reference: DDR5 laptop ~40-60 GB/s, LPDDR5x ~50-70 GB/s\n")

print(f"  {'Path':<25} {'Model':<6} {'Device':<8} {'Latency(ms)':>12} {'Weight(GB)':>10} {'BW(GB/s)':>10}")
print(f"  {'-'*25} {'-'*6} {'-'*8} {'-'*12} {'-'*10} {'-'*10}")

for label, data, bits in [
    ("MNB 4-bit", df[(df["format_type"]=="mnb") & (df["bits"]==4) & (df["symmetry"]=="sym") & (df["weight_layout"]=="original") & (df["seq_length"]==1)], 4),
    ("MNB 8-bit", df[(df["format_type"]=="mnb") & (df["bits"]==8) & (df["symmetry"]=="sym") & (df["weight_layout"]=="original") & (df["seq_length"]==1)], 8),
]:
    for _, row in data.iterrows():
        ws = WEIGHT_SIZES_GB.get((row["model_size"], bits), 0)
        bw = ws / (row["mean_ms"] / 1000) if row["mean_ms"] > 0 else 0
        print(f"  {label:<25} {row['model_size']:<6} {row['device']:<8} {row['mean_ms']:>12.1f} {ws:>10.2f} {bw:>10.1f}")

print(f"\n  Unfused 4-bit DQ bandwidth (how far from optimal):")
uf_n = df[(df["format_type"]=="qdq") & (df["bits"]==4) & (df["granularity"]=="block") & (df["scenario"]=="qdq_unfused") & (df["weight_layout"]=="original") & (df["symmetry"]=="asym") & (df["signedness"]=="signed") & (df["seq_length"]==1)]
for _, row in uf_n.iterrows():
    ws = WEIGHT_SIZES_GB.get((row["model_size"], 4), 0)
    bw = ws / (row["mean_ms"] / 1000) if row["mean_ms"] > 0 else 0
    print(f"  {'QDQ 4-bit unfused':<25} {row['model_size']:<6} {row['device']:<8} {row['mean_ms']:>12.1f} {ws:>10.2f} {bw:>10.1f}")

### N. Observations

MNB at decode achieves bandwidth close to practical memory bandwidth limits on each device.
This means MNB is already near-optimal for decode. Further speedups require reducing model size.

Unfused DQ effective bandwidth is extremely low (sub-1 GB/s), confirming it's purely compute-bound
on the scalar dequantization loop, not memory-bound. Threading + SIMD would directly improve this.

## O. Practical Recommendation Matrix

Based on all the analysis above, here are concrete recommendations for each device and workload type.

In [ ]:
# O: Recommendation matrix
print("  O. Recommended Configuration per Device x Workload")
print("  " + "="*100)
print()
recs = [
    ["AMD x86",  "Decode-heavy (seq=1)",   "MNB 4-bit (sym or asym). ~7-98ms for 0.5b-7b. QDQ 4-bit block fused equivalent."],
    ["AMD x86",  "Prefill (long ctx)",      "MNB 4-bit or 8-bit. 8-bit 0-15% slower at seq=1024. Choose by accuracy."],
    ["AMD x86",  "QDQ-only (no MNB)",       "8-bit block unsigned. 3-5x faster than 4-bit unfused at decode."],
    ["Intel x86","Decode-heavy (seq=1)",    "MNB 4-bit. Intel has worst unfused penalty. Always fuse or use MNB."],
    ["Intel x86","Prefill (long ctx)",      "MNB 4-bit or 8-bit. 8-bit 0-12% faster for 7b at prefill."],
    ["Intel x86","QDQ-only (no MNB)",       "8-bit block unsigned. 4-5x faster than 4-bit unfused. Intel DQ weakest."],
    ["ARM",      "Decode-heavy (seq=1)",    "MNB 4-bit SYMMETRIC. Asym ~50% slower for >=1.5b (ARM-specific)."],
    ["ARM",      "Prefill (long ctx)",      "MNB 8-bit sym. ARM 8-bit faster than 4-bit at seq>=128. Except 7b."],
    ["ARM",      "Large models",            "Avoid 7b 8-bit on 16GB devices. Exceeds RAM -> 1.6-4.3x slowdown."],
    ["ARM",      "QDQ-only (no MNB)",       "8-bit block preferred. ARM nibble overhead lower but still significant."],
]

print(f"  {'Device':<12} {'Workload':<24} {'Recommendation'}")
print(f"  {'-'*12} {'-'*24} {'-'*70}")
for dev, wl, rec in recs:
    print(f"  {dev:<12} {wl:<24} {rec}")

print()
print("  UNIVERSAL RULES:")
print("  * Always use block quantization over channel (3-7x faster for 8-bit at decode)")
print("  * Always use original layout -- transpose breaks fusion and adds overhead")
print("  * If unfused QDQ: prefer unsigned (8-10% faster for 4-bit on x86)")
print("  * Never use channel-transpose: extreme load times (40-228s) and poor latency")
print("  * QDQ fused = MNB native for 4-bit block original -- no perf reason to prefer one")

### O.2 Breakeven Analysis: MNB Load Cost vs QDQ Unfused

MNB/fused QDQ has higher load time (PrePack: 1-15s) but much faster inference.
QDQ unfused has near-zero load time but slower inference. How many inferences until MNB pays off?

In [ ]:
# O.2: Breakeven analysis
load_data_o = df.drop_duplicates(
    subset=["device", "format_type", "model_size", "bits", "symmetry",
            "granularity", "signedness", "weight_layout", "scenario"])
fused_ld = load_data_o[
    (load_data_o["format_type"]=="qdq") & (load_data_o["bits"]==4) & (load_data_o["granularity"]=="block") &
    (load_data_o["scenario"]=="qdq_fused") & (load_data_o["weight_layout"]=="original") &
    (load_data_o["symmetry"]=="asym") & (load_data_o["signedness"]=="signed")
][["device","model_size","model_load_time_s"]].rename(columns={"model_load_time_s": "fused_load_s"})
unfused_ld = load_data_o[
    (load_data_o["format_type"]=="qdq") & (load_data_o["bits"]==4) & (load_data_o["granularity"]=="block") &
    (load_data_o["scenario"]=="qdq_unfused") & (load_data_o["weight_layout"]=="original") &
    (load_data_o["symmetry"]=="asym") & (load_data_o["signedness"]=="signed")
][["device","model_size","model_load_time_s"]].rename(columns={"model_load_time_s": "unfused_load_s"})
lds = fused_ld.merge(unfused_ld, on=["device","model_size"])
lds["prepack_cost_ms"] = (lds["fused_load_s"] - lds["unfused_load_s"]) * 1000

print("  O.2 Breakeven: How many inferences for fused to pay off?")
print("  " + "="*80)
print(f"  {'device':<8} {'model':<6} {'prepack(ms)':>12} {'seq=1 save':>12} {'breakeven':>10} {'seq=1024 save':>14} {'breakeven':>10}")
print(f"  {'-'*8} {'-'*6} {'-'*12} {'-'*12} {'-'*10} {'-'*14} {'-'*10}")
for dev in ["amd", "intel"]:
    for ms in MODEL_SIZE_ORDER:
        lr = lds[(lds["device"]==dev) & (lds["model_size"]==ms)]
        if lr.empty: continue
        pp = lr.iloc[0]["prepack_cost_ms"]
        fl = df[(df["format_type"]=="qdq") & (df["bits"]==4) & (df["granularity"]=="block") & (df["scenario"]=="qdq_fused") & (df["weight_layout"]=="original") & (df["symmetry"]=="asym") & (df["signedness"]=="signed") & (df["device"]==dev) & (df["model_size"]==ms)]
        ul = df[(df["format_type"]=="qdq") & (df["bits"]==4) & (df["granularity"]=="block") & (df["scenario"]=="qdq_unfused") & (df["weight_layout"]=="original") & (df["symmetry"]=="asym") & (df["signedness"]=="signed") & (df["device"]==dev) & (df["model_size"]==ms)]
        f1 = fl[fl["seq_length"]==1]["mean_ms"]; u1 = ul[ul["seq_length"]==1]["mean_ms"]
        f1024 = fl[fl["seq_length"]==1024]["mean_ms"]; u1024 = ul[ul["seq_length"]==1024]["mean_ms"]
        if f1.empty or u1.empty: continue
        s1 = u1.iloc[0] - f1.iloc[0]; be1 = pp / s1 if s1 > 0 else float('inf')
        s1024 = u1024.iloc[0] - f1024.iloc[0] if not u1024.empty else 0
        be1024 = pp / s1024 if s1024 > 0 else float('inf')
        print(f"  {dev:<8} {ms:<6} {pp:>12.0f} {s1:>12.0f} {be1:>10.1f} {s1024:>14.0f} {be1024:>10.1f}")
print(f"\n  Breakeven < 1 = fused pays off after a single inference.")
print(f"  At decode (seq=1): fused always pays off immediately.")
print(f"  At prefill (seq=1024): fused pays off after 1-15 inferences.")

## Summary

**Fused QDQ and native MNB have the same performance.** ORT's `DQMatMulToMatMulNBits` pass rewrites 4-bit block QDQ into the MatMulNBits kernel, so they end up running identical code. Latencies match across all three devices and all model sizes (A).

**The unfused DQ path is the bottleneck.** At decode (seq=1), unfused is 80-209x slower than fused. At prefill (seq=1024), the gap shrinks to 1.0-2.9x because compute dominates and the DQ cost gets amortized.

**Fusion only works for one configuration:** 4-bit, block-quantized, original (non-transposed) layout. Everything else falls through to the slow unfused path (E):
- A Transpose node between DQ and MatMul blocks the graph rewrite
- Per-channel quantization has no rewrite rule (selector requires `block_size` and `axis=0`)
- The fusion selector (`DQMatMulNodeGroupSelector`) has an `Is4BitIntType` check that rejects 8-bit, even though the MatMulNBits op itself supports 2/4/8 bits

**Sym vs asym:** no meaningful difference on x86. On ARM, asym is ~50% slower for 4-bit models >= 1.5b. Too consistent across seq lengths to be noise, probably a zero-point handling difference in the ARM kernel (B).

**4-bit vs 8-bit (MNB):** 4-bit is 1.4-2.0x faster at decode on x86 (bandwidth-bound, half the weight size). At seq >= 128 the gap disappears (compute-bound). ARM follows the same trend for models that fit in RAM (C).

**Signed vs unsigned:** no difference when fused (same kernel). Unfused 4-bit DQ is 8-10% faster with unsigned on x86, likely due to the nibble extraction code path (D).

**Block vs channel (unfused):** block is faster at decode. For 4-bit: 7-58% faster depending on device. For 8-bit: 3-7x faster. Gap shrinks at prefill. AMD crosses over at long sequences where channel becomes slightly faster (F).

**Unfused 8-bit DQ is faster than 4-bit on x86.** At decode, unfused 8-bit is 3-5x faster than unfused 4-bit (H). The scalar nibble extraction in the 4-bit DQ kernel is expensive enough to wipe out the bandwidth advantage of smaller weights. ARM shows the opposite: its nibble unpacking is cheaper, so 4-bit wins there (G).

**Load time** is driven by PrePack (weight repacking). Only MNB and fused 4-bit QDQ trigger it. All other QDQ paths load in under 0.5s regardless of model size (I).

### What could be improved in the unfused DQ path

- **Add threading to the blocked sub-byte DQ path** (low effort, ~4-8x). The non-sub-byte path already has threading. The sub-byte path ignores the thread pool (`ORT_UNUSED_PARAMETER(thread_pool)` at quantize_linear.cc:452). No data dependencies between blocks.
- **Add SIMD nibble extraction** (medium effort, ~4-8x on top of threading). AVX2 and NEON patterns for this already exist in MLAS (`sqnbitgemm_kernel_avx2.cpp`, `sqnbitgemm_kernel_neon_fp32.cpp`).
- **Add `PrePack()` for constant weights** (medium effort). DequantizeLinear is a plain OpKernel with no PrePack override. It re-dequantizes every inference call. For LLM weights (always constant), dequantizing once at load time and caching the result would eliminate the per-call DQ cost.
- **Remove the `ORT_CLIENT_PACKAGE_BUILD` gate for int8 DQ threading** (low effort). The threading code exists at quantize_linear.cc:340 but is behind a build flag with a TODO to make it the default.
- **Extend the fusion selector to support 8-bit.** The MatMulNBits kernel already handles 8-bit (`nbits_ == 8` at matmul_nbits.cc:124). The blocker is the `Is4BitIntType` check in `DQMatMulNodeGroupSelector` (qdq_selectors.cc:588). Removing that gate (and handling the weight repacking) would let 8-bit QDQ fuse too.

Threading + SIMD combined would close most of the 80-150x gap. PrePack would eliminate it for the common case of constant weights.

### Additional optimization opportunities (from sections K-O)

- **Extend fusion to 8-bit** (high impact, low-medium effort): Section A.2 shows the 8-bit MNB vs QDQ gap. The `Is4BitIntType` check in `DQMatMulNodeGroupSelector` (`qdq_selectors.cc:588`) rejects 8-bit, but `MatMulNBits` already supports it (`nbits_ == 8` at `matmul_nbits.cc:124`). Removing this gate would close the 8-bit gap.
- **Investigate ARM asym penalty** (medium effort): Section M shows ARM 4-bit asym is ~50% slower than sym for models >=1.5b, consistent across all seq lengths. Not seen on x86. Root cause is likely in the ARM kernel's zero-point handling — could be a missed NEON optimization or a branch misprediction issue.
- **Model preprocessing to remove Transpose nodes** (low effort for users): Section E shows transpose breaks fusion. If models come with Transpose between DQ and MatMul, a preprocessing step to fold/remove it would restore fusion eligibility.
- **Benchmark `accuracy_level=4` (CompInt8)** (not measured): MatMulNBits with CompInt8 uses integer GEMM (`SQNBIT_CompInt8`) which avoids FP32 dequant entirely on x86. This is expected to be the fastest MNB path. Only needs a session option change — quick to test.

### Device-specific summary

**AMD**: Most predictable device. Clean linear scaling. Sym/asym parity. MNB 4-bit is optimal for decode. At prefill, 4-bit and 8-bit are comparable. AMD 1.5b has some measurement outliers worth re-running.

**Intel**: Highest unfused DQ penalty (up to 209x vs MNB at decode). Benefits most from DQ kernel optimization (threading/SIMD). Otherwise similar to AMD for fused/MNB paths.

**ARM**: Three unique issues — (1) asym 50% slower for 4-bit >=1.5b (always prefer sym), (2) 8-bit MNB faster than 4-bit at prefill (nibble unpacking is expensive on ARM), (3) 7b 8-bit exceeds 16GB RAM. ARM data has higher variance overall; re-run with 100+ iterations recommended.

### Data quality issues

- **ARM 7b 8-bit** results are not usable. The model exceeds 16 GB device RAM, causing swapping. Shows up as erratic numbers in C, F, and H.
- **AMD 1.5b** has outliers in the sym/asym and 4-bit/8-bit comparisons (B, C) that don't appear on Intel. Probably measurement noise, worth re-running with more iterations.
- **ARM 0.5b 4-bit sym** at seq=1 reports 29ms vs 10ms for asym. This is the opposite of the trend for other ARM models and looks like a bad run (B).
- **`channel-sym-signed-transpose`** models fail to load on all three devices. Likely an ORT bug (E).